# Unified Benchmark — Symbolic KAN · Exact Reconstruction · FastKAN · EfficientKAN
### One notebook, one trained model per seed, all datasets

This notebook fixes the apparent inconsistency between the symbolic KAN and reconstruction by deriving
**both from the same trained model**:

1. Train a real **pykan KAN**, then `auto_symbolic()` it → the **Symbolic KAN** (framework forward pass).
2. Extract its symbolic formula with `symbolic_formula()` and evaluate it directly →
   **Exact Reconstruction**. Because it is the *same* formula re-associated, its output matches the
   symbolic model to machine precision (logit gap $\sim10^{-13}$).
3. Train **FastKAN** and **EfficientKAN** on the same split for comparison.

So "Symbolic" and "Reconstruction" now share identical accuracy by construction — the difference is purely
**latency** (framework dispatch vs direct formula evaluation). Reported as mean ± std over seeds, per
dataset.

Datasets: NSL-KDD, Pima, Covertype (hard), MNIST — each loads only if available.

## 0. Setup

In [1]:
# !pip install pykan torch scikit-learn numpy pandas sympy --quiet
# !curl -sL -o fastkan.py https://raw.githubusercontent.com/ZiyaoLi/fast-kan/master/fastkan/fastkan.py
# !curl -sL -o efficient_kan.py https://raw.githubusercontent.com/Blealtan/efficient-kan/master/src/efficient_kan/kan.py
import numpy as np, pandas as pd, torch, time, os, glob, warnings, contextlib, io, sympy
warnings.filterwarnings("ignore")
import torch.nn.functional as Fcn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.datasets import fetch_openml, load_digits
from kan import KAN
import fastkan, efficient_kan
torch.set_default_dtype(torch.float64)

SEEDS   = [0, 1, 2]        # raise to 5 for final numbers (pykan is slow)
BATCH   = 256
STEPS   = 20               # pykan LBFGS steps
HIDDEN  = 2                # KAN hidden width (small -> tractable symbolic formula)
SYM_LIB = ['x','x^2','x^3']   # polynomial library (CKKS-compatible)
EK_EPOCHS = 60
print("ready | seeds:", SEEDS)

ready | seeds: [0, 1, 2]


## 1. Datasets (same loaders as before)

In [2]:
MAX_ROWS = 6000
NSL_COLS = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate","label"]

def load_nsl():
    cands=(glob.glob("KDDTrain+*.txt")+glob.glob("KDDTrain+*.csv")
           +glob.glob("KDDTest+*.txt")+glob.glob("KDDTest+*.csv")+glob.glob("KDDTrain*.txt"))
    path=next((c for c in cands if os.path.isfile(c)),None)
    if path is None: print("NSL-KDD not found -> skipped"); return None
    raw=pd.read_csv(path,header=None); n=raw.shape[1]
    raw=raw.iloc[:,:42] if n==43 else (raw if n==42 else raw.iloc[:,:min(n,42)])
    raw.columns=NSL_COLS[:raw.shape[1]]
    if len(raw)>MAX_ROWS: raw=raw.sample(MAX_ROWS,random_state=0)
    y=(raw["label"].astype(str).str.lower()!="normal").astype(int).values
    df=raw.drop(columns=["label"])
    for c in ["protocol_type","service","flag"]:
        if c in df.columns: df[c]=pd.factorize(df[c])[0]
    X=df.apply(pd.to_numeric,errors="coerce").fillna(0).values.astype(float)
    print(f"NSL-KDD from '{path}': {X.shape}"); return X,y,"NSL-KDD"

def load_pima():
    try:
        d=fetch_openml("diabetes",version=1,as_frame=True,parser="auto")
        y=(d.target.astype(str)=="tested_positive").astype(int).values
        X=d.data.apply(pd.to_numeric,errors="coerce").dropna(axis=1,how="all").values.astype(float)
        return X,y,"Pima Diabetes"
    except Exception as e: print("Pima skipped:",str(e)[:40]); return None

def load_covtype():
    try:
        from sklearn.datasets import fetch_covtype
        d=fetch_covtype(); idx=np.random.RandomState(0).choice(len(d.data),size=min(MAX_ROWS,len(d.data)),replace=False)
        return d.data[idx].astype(float),(d.target[idx]==2).astype(int),"Covertype (hard)"
    except Exception as e: print("Covertype skipped:",str(e)[:40]); return None

def load_mnist():
    try:
        d=fetch_openml("mnist_784",version=1,as_frame=False,parser="auto")
        idx=np.random.RandomState(0).choice(len(d.data),size=min(MAX_ROWS,len(d.data)),replace=False)
        return d.data[idx].astype(float)/255.0,(d.target[idx].astype(int)>=5).astype(int),"MNIST (image)"
    except Exception as e:
        print("MNIST -> Digits 8x8:",str(e)[:40])
        d=load_digits(); return d.data.astype(float)/16.0,(d.target>=5).astype(int),"Digits 8x8 (image)"

DATASETS=[]
for loader in [load_nsl, load_pima, load_covtype, load_mnist]:
    r=loader()
    if r is None: continue
    X_,y_,nm=r; DATASETS.append((nm,X_,y_)); print(f"  -> {nm}: {X_.shape}, pos {y_.mean():.3f}")
print("datasets:", [d[0] for d in DATASETS])

NSL-KDD from 'KDDTest+.txt': (6000, 41)
  -> NSL-KDD: (6000, 41), pos 0.553
  -> Pima Diabetes: (768, 8), pos 0.349
  -> Covertype (hard): (6000, 54), pos 0.483
  -> MNIST (image): (6000, 784), pos 0.489
datasets: ['NSL-KDD', 'Pima Diabetes', 'Covertype (hard)', 'MNIST (image)']


## 2. Core: train KAN, symbolify, reconstruct from the SAME formula

In [3]:
def metrics(yte, logits):
    p=torch.softmax(torch.tensor(np.asarray(logits)),1).numpy(); pred=p.argmax(1)
    try: auc=roc_auc_score(yte,p[:,1])
    except ValueError: auc=float("nan")
    return dict(AUC=auc, Acc=accuracy_score(yte,pred), F1=f1_score(yte,pred,zero_division=0))

def bench_np(fn, X):
    Xb=X[:BATCH]
    for _ in range(5): fn(Xb)
    a=[]
    for _ in range(30): t=time.perf_counter(); fn(Xb); a.append(time.perf_counter()-t)
    return np.mean(a)*1000

def bench_torch(m, X):
    Xb=torch.tensor(X[:BATCH])
    with torch.no_grad():
        for _ in range(5): m(Xb)
        a=[]
        for _ in range(30): t=time.perf_counter(); m(Xb); a.append(time.perf_counter()-t)
    return np.mean(a)*1000

def bench_kan_symbolic(model, X):
    Xb=torch.tensor(X[:BATCH])
    with torch.no_grad():
        for _ in range(3): model(Xb)
        a=[]
        for _ in range(20): t=time.perf_counter(); model(Xb); a.append(time.perf_counter()-t)
    return np.mean(a)*1000

def train_kan_and_reconstruct(Xtr, ytr, Xte, yte, seed):
    F=Xtr.shape[1]
    Xtr_t=torch.tensor(Xtr); Xte_t=torch.tensor(Xte)
    with contextlib.redirect_stdout(io.StringIO()):
        model=KAN(width=[F,HIDDEN,2], grid=4, k=3, seed=seed)
        ds={'train_input':Xtr_t,'train_label':torch.tensor(ytr).long(),
            'test_input':Xte_t,'test_label':torch.tensor(yte).long()}
        model.fit(ds, opt="LBFGS", steps=STEPS, loss_fn=lambda p,l:Fcn.cross_entropy(p,l.long()))
        model.auto_symbolic(lib=SYM_LIB)
        formulas, xvars = model.symbolic_formula()
    # symbolic (framework) metrics + latency
    with torch.no_grad(): lg_sym=model(Xte_t).numpy()
    sym_m=metrics(yte, lg_sym); sym_ms=bench_kan_symbolic(model, Xte)
    sym_par=int(sum(p.numel() for p in model.parameters()))
    # reconstruction: evaluate the SAME formula directly
    funcs=[sympy.lambdify(xvars, f, 'numpy') for f in formulas]
    def recon(Xin):
        cols=[np.broadcast_to(np.asarray(funcs[k](*[Xin[:,j] for j in range(F)]), float),
                              (Xin.shape[0],)) for k in range(len(funcs))]
        return np.stack(cols, 1)
    lg_rec=recon(Xte); rec_m=metrics(yte, lg_rec); rec_ms=bench_np(recon, Xte)
    gap=float(np.abs(lg_sym - lg_rec).mean())
    # count reconstruction params = number of nonzero symbolic coefficients
    rec_par=sum(len(sympy.Poly(f, *xvars).coeffs()) if f.free_symbols else 1 for f in formulas)
    return dict(sym=(sym_m,sym_ms,sym_par), rec=(rec_m,rec_ms,int(rec_par)), gap=gap)
print("KAN+reconstruction runner ready")

KAN+reconstruction runner ready


## 3. FastKAN / EfficientKAN on the same split

In [4]:
def train_fastkan(Xtr, ytr, Xte, yte):
    F=Xtr.shape[1]
    m=fastkan.FastKAN([F,5,2])
    opt=torch.optim.Adam(m.parameters(),lr=0.01)
    Xt,yt=torch.tensor(Xtr),torch.tensor(ytr).long()
    for _ in range(EK_EPOCHS): opt.zero_grad(); Fcn.cross_entropy(m(Xt),yt).backward(); opt.step()
    with torch.no_grad(): lg=m(torch.tensor(Xte)).numpy()
    return metrics(yte,lg), bench_torch(m,Xte), int(sum(p.numel() for p in m.parameters()))

def train_effkan(Xtr, ytr, Xte, yte):
    F=Xtr.shape[1]
    m=efficient_kan.KAN([F,5,2])
    opt=torch.optim.Adam(m.parameters(),lr=0.01)
    Xt,yt=torch.tensor(Xtr),torch.tensor(ytr).long()
    for _ in range(EK_EPOCHS): opt.zero_grad(); Fcn.cross_entropy(m(Xt),yt).backward(); opt.step()
    with torch.no_grad(): lg=m(torch.tensor(Xte)).numpy()
    return metrics(yte,lg), bench_torch(m,Xte), int(sum(p.numel() for p in m.parameters()))
print("FastKAN / EfficientKAN trainers ready")

FastKAN / EfficientKAN trainers ready


## 4. Run everything

In [5]:
records=[]; gaps=[]
for ds_name, X_raw, y_raw in DATASETS:
    for seed in SEEDS:
        np.random.seed(seed); torch.manual_seed(seed)
        Xtr,Xte,ytr,yte=train_test_split(X_raw,y_raw,test_size=0.3,random_state=seed,stratify=y_raw)
        sc=StandardScaler().fit(Xtr); Xtr,Xte=sc.transform(Xtr),sc.transform(Xte)
        try:
            r=train_kan_and_reconstruct(Xtr,ytr,Xte,yte,seed)
            sm,sms,spar=r["sym"]; rm,rms,rpar=r["rec"]; gaps.append(r["gap"])
            records.append(dict(Dataset=ds_name,seed=seed,Method="Symbolic KAN (framework)",**sm,ms_batch=sms,Params=spar))
            records.append(dict(Dataset=ds_name,seed=seed,Method="Reconstruction (ours)",**rm,ms_batch=rms,Params=rpar))
            fm,fms,fpar=train_fastkan(Xtr,ytr,Xte,yte)
            records.append(dict(Dataset=ds_name,seed=seed,Method="FastKAN",**fm,ms_batch=fms,Params=fpar))
            em,ems,epar=train_effkan(Xtr,ytr,Xte,yte)
            records.append(dict(Dataset=ds_name,seed=seed,Method="EfficientKAN",**em,ms_batch=ems,Params=epar))
            print(f"{ds_name} s{seed}: sym AUC {sm['AUC']:.4f} ({sms:.1f}ms) | recon AUC {rm['AUC']:.4f} ({rms:.3f}ms) "
                  f"| gap {r['gap']:.1e} | FastKAN {fm['AUC']:.4f} | EffKAN {em['AUC']:.4f}")
        except Exception as e:
            print(f"{ds_name} s{seed}: FAILED {str(e)[:70]}")
df=pd.DataFrame(records)
print(f"\nrows: {len(df)} | mean symbolic<->reconstruction logit gap: {np.mean(gaps):.2e} (≈0 confirms exactness)")

| train_loss: 2.03e-01 | test_loss: 9.40e-01 | reg: 1.40e+02 | : 100%|█| 20/20 [00:23<00:00,  1.19s/
description:   0%|                                                           | 0/20 [00:00<?, ?it/s]

NSL-KDD s0: sym AUC 0.8514 (12.2ms) | recon AUC 0.8514 (0.196ms) | gap 3.5e-13 | FastKAN 0.9944 | EffKAN 0.9855


| train_loss: 1.84e-01 | test_loss: 6.20e-01 | reg: 9.81e+01 | : 100%|█| 20/20 [00:21<00:00,  1.05s/
description:   0%|                                                           | 0/20 [00:00<?, ?it/s]

NSL-KDD s1: sym AUC 0.8258 (12.5ms) | recon AUC 0.8258 (0.234ms) | gap 6.7e-14 | FastKAN 0.9943 | EffKAN 0.9867


| train_loss: 2.12e-01 | test_loss: 4.77e-01 | reg: 1.06e+02 | : 100%|█| 20/20 [00:22<00:00,  1.10s/
description:   0%|                                                           | 0/20 [00:00<?, ?it/s]

...

MNIST (image) s0: sym AUC 0.8772 (766.0ms) | recon AUC 0.8772 (22.116ms) | gap 4.0e-13 | FastKAN 0.9320 | EffKAN 0.9600


| train_loss: 6.50e-10 | test_loss: 4.03e+00 | reg: 5.47e+02 | : 100%|█| 20/20 [08:31<00:00, 25.60s/
description:   0%|                                                           | 0/20 [00:00<?, ?it/s]

MNIST (image) s1: sym AUC 0.8482 (134.9ms) | recon AUC 0.8482 (5.764ms) | gap 1.5e-13 | FastKAN 0.9324 | EffKAN 0.9583


| train_loss: 7.43e-08 | test_loss: 5.83e+00 | reg: 9.66e+02 | : 100%|█| 20/20 [04:45<00:00, 14.26s/


MNIST (image) s2: sym AUC 0.8838 (136.4ms) | recon AUC 0.8838 (6.077ms) | gap 3.6e-13 | FastKAN 0.9331 | EffKAN 0.9690

rows: 48 | mean symbolic<->reconstruction logit gap: 1.14e-13 (≈0 confirms exactness)


## 5. Summary per dataset (mean ± std)\nSymbolic and Reconstruction now share accuracy by construction; the difference is latency.

In [6]:
ORDER=["Symbolic KAN (framework)","Reconstruction (ours)","FastKAN","EfficientKAN"]
for ds in df.Dataset.unique():
    sub=df[df.Dataset==ds]; g=sub.groupby("Method")
    out=pd.DataFrame(index=[m for m in ORDER if m in g.groups])
    for c in ["AUC","Acc","F1","ms_batch"]:
        out[c]=g[c].mean().round(4).astype(str)+" ± "+g[c].std().round(4).astype(str)
    out["Params"]=g["Params"].mean().round(0).astype(int)
    out["recon speedup"]=""
    if "Symbolic KAN (framework)" in g.groups and "Reconstruction (ours)" in g.groups:
        sp=g.get_group("Symbolic KAN (framework)")["ms_batch"].mean()/g.get_group("Reconstruction (ours)")["ms_batch"].mean()
        out.loc["Reconstruction (ours)","recon speedup"]=f"{sp:.0f}x vs symbolic"
    print(f"\n=== {ds} (mean ± std over {len(SEEDS)} seeds) ==="); display(out.loc[[m for m in ORDER if m in out.index]])


=== NSL-KDD (mean ± std over 3 seeds) ===


,AUC,Acc,F1,ms_batch,Params,recon speedup
Symbolic KAN (framework),0.8294 ± 0.0205,0.6794 ± 0.1274,0.7392 ± 0.059,12.6411 ± 0.545,1779,
Reconstruction (ours),0.8294 ± 0.0205,0.6794 ± 0.1274,0.7392 ± 0.059,0.2175 ± 0.0197,100,58x vs symbolic
FastKAN,0.9927 ± 0.003,0.9611 ± 0.0025,0.9653 ± 0.0022,1.0249 ± 0.1242,2050,
EfficientKAN,0.9846 ± 0.0027,0.9509 ± 0.0051,0.9565 ± 0.0045,3.9649 ± 0.1289,2150,



=== Pima Diabetes (mean ± std over 3 seeds) ===


,AUC,Acc,F1,ms_batch,Params,recon speedup
Symbolic KAN (framework),0.8135 ± 0.0452,0.7605 ± 0.0261,0.5756 ± 0.085,5.7969 ± 1.2699,426,
Reconstruction (ours),0.8135 ± 0.0452,0.7605 ± 0.0261,0.5756 ± 0.085,0.0456 ± 0.0026,20,127x vs symbolic
FastKAN,0.8094 ± 0.0127,0.759 ± 0.0125,0.6176 ± 0.0209,0.5794 ± 0.075,499,
EfficientKAN,0.8188 ± 0.0161,0.746 ± 0.0214,0.6139 ± 0.034,1.6027 ± 0.0298,500,



=== Covertype (hard) (mean ± std over 3 seeds) ===


,AUC,Acc,F1,ms_batch,Params,recon speedup
Symbolic KAN (framework),0.7065 ± 0.054,0.6246 ± 0.0631,0.5494 ± 0.1085,16.3162 ± 2.3045,2312,
Reconstruction (ours),0.7065 ± 0.054,0.6246 ± 0.0631,0.5494 ± 0.1085,0.1976 ± 0.0086,113,83x vs symbolic
FastKAN,0.8535 ± 0.0063,0.7785 ± 0.0148,0.7697 ± 0.0195,1.1524 ± 0.1484,2661,
EfficientKAN,0.8516 ± 0.0083,0.777 ± 0.0162,0.7759 ± 0.0177,4.5804 ± 0.4699,2800,



=== MNIST (image) (mean ± std over 3 seeds) ===


,AUC,Acc,F1,ms_batch,Params,recon speedup
Symbolic KAN (framework),0.8697 ± 0.0189,0.7924 ± 0.0652,0.7628 ± 0.0898,345.7745 ± 363.9382,32242,
Reconstruction (ours),0.8697 ± 0.0189,0.7924 ± 0.0652,0.7628 ± 0.0898,11.3191 ± 9.3518,1929,31x vs symbolic
FastKAN,0.9325 ± 0.0006,0.8709 ± 0.0046,0.8676 ± 0.0036,10.1737 ± 10.5579,36971,
EfficientKAN,0.9624 ± 0.0057,0.908 ± 0.0043,0.906 ± 0.0044,67.4427 ± 77.8788,39300,


## 6. Notes

- **Same model, two evaluations.** Symbolic KAN and Reconstruction are the *same* trained network: the
  symbolic forward pass through pykan, versus direct evaluation of its extracted `symbolic_formula()`.
  Their accuracy is identical (logit gap $\sim10^{-13}$); only latency differs. This is the exactness
  claim, demonstrated rather than asserted.
- **FastKAN / EfficientKAN** are separate models trained on the same split, for the speed/accuracy
  comparison.
- pykan symbolic training is slow; `HIDDEN=2`, `STEPS=20`, `MAX_ROWS=6000` keep it tractable. Raise
  `SEEDS` to 5 and `STEPS` for final numbers. If MNIST is too slow, reduce `MAX_ROWS` first.
- The reconstruction here evaluates the real pykan formula, so on high-dimensional inputs (MNIST) the
  symbolic formula can be large; this is expected and matches the paper's note that the advantage is
  strongest for compact tabular models.